# COMP 3610 – Assignment 1  
## NYC Yellow Taxi Data Analysis (January 2024)

This notebook walks through the full data pipeline for the NYC Yellow Taxi dataset.  
It includes data ingestion, cleaning, SQL analysis using DuckDB and insights derived from the results.  

The goal of this analysis is to understand the trip patterns, fare behavior and payment usage trends in NYC during January 2024.

## Part 1: Data Ingestion

In this section, I download the January 2024 NYC Yellow Taxi trip data and the taxi zone lookup file.  
The raw data is stored locally inside a structured folder to separate raw and processed files.  

This ensures the pipeline is reproducible and organized.


In [3]:
import os
import requests

# Create folder for raw data
os.makedirs("data/raw", exist_ok=True)

trip_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
zone_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

trip_path = "data/raw/yellow_tripdata_2024-01.parquet"
zone_path = "data/raw/taxi_zone_lookup.csv"

# Download trip data
if not os.path.exists(trip_path):
    print("Downloading trip data...")
    r = requests.get(trip_url)
    with open(trip_path, "wb") as f:
        f.write(r.content)
    print("Trip data downloaded.")
else:
    print("Trip file already exists.")

# Download zone lookup
if not os.path.exists(zone_path):
    print("Downloading zone lookup...")
    r = requests.get(zone_url)
    with open(zone_path, "wb") as f:
        f.write(r.content)
    print("Zone lookup downloaded.")
else:
    print("Zone file already exists.")


Trip data downloaded.
Zone lookup downloaded.


## Data Validation

Before cleaning the dataset, I verified that the required columns exist and that the datetime fields are correctly formatted.


In [4]:
import polars as pl

df = pl.read_parquet("data/raw/yellow_tripdata_2024-01.parquet")

print("Dataset loaded.")
print("Total rows:", df.height)
print("Total columns:", df.width)


Dataset loaded.
Total rows: 2964624
Total columns: 19


In [5]:
required_columns = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "payment_type"
]

missing = [col for col in required_columns if col not in df.columns]

if len(missing) > 0:
    print("Missing columns:", missing)
else:
    print("All required columns are present.")


All required columns are present.


In [6]:
print("Pickup datetime type:", df.schema["tpep_pickup_datetime"])
print("Dropoff datetime type:", df.schema["tpep_dropoff_datetime"])


Pickup datetime type: Datetime(time_unit='ns', time_zone=None)
Dropoff datetime type: Datetime(time_unit='ns', time_zone=None)


## Data Cleaning

The raw dataset contains millions of records, hence,
before analysis, I removed invalid or inconsistent entries to ensure accuracy.

The cleaning steps included:
Removing rows with missing critical values (pickup/dropoff time, location IDs, fare, distance)
Filtering out trips with negative fares or zero/negative distances
Removing trips where dropoff time occurred before pickup time


In [7]:
import polars as pl

df = pl.read_parquet("data/raw/yellow_tripdata_2024-01.parquet")

start_rows = df.height
print("Starting rows:", start_rows)

#Remove rows with null pickup/dropoff/location/fare fields
critical_cols = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "fare_amount",
    "trip_distance"
]

df1 = df.drop_nulls(critical_cols)
print("Removed null critical fields:", start_rows - df1.height)

#Remove invalid trips:
#distance<= 0, negative fare,fare > 500
df2 = df1.filter(
    (pl.col("trip_distance") > 0) &
    (pl.col("fare_amount") >= 0) &
    (pl.col("fare_amount") <= 500)
)
print("Removed invalid distance/fare:", df1.height - df2.height)

#Remove trips where dropoff less than pickup
df_clean = df2.filter(pl.col("tpep_dropoff_datetime") >= pl.col("tpep_pickup_datetime"))
print("Removed dropoff before pickup:", df2.height - df_clean.height)

print("Final cleaned rows:", df_clean.height)


Starting rows: 2964624
Removed null critical fields: 0
Removed invalid distance/fare: 94466
Removed dropoff before pickup: 56
Final cleaned rows: 2870102


## Observations During Cleaning

A significant number of trips had ivalid fare or distance values.
Very few trips had dropoff times earlier than pickup times.
The majority of records were valid, indicating overall good data quality.

After cleaning, the dataset contained approximately 2.87 million valid trips.


## Feature Engineering

To enhance the dataset, I created additional features:

Trip duration (in minutes)
Trip speed (in miles per hour)
Pickup hour
Pickup day of week

These engineered features allow better time-based and behavioral analysis.


In [8]:
df_feat = df_clean.with_columns([
    #the trip duration in mins
    ((pl.col("tpep_dropoff_datetime") - pl.col("tpep_pickup_datetime"))
     .dt.total_seconds() / 60).alias("trip_duration_minutes"),

    #speed in mph
    (pl.col("trip_distance") / (((pl.col("tpep_dropoff_datetime") - pl.col("tpep_pickup_datetime"))
     .dt.total_seconds() / 3600))).alias("trip_speed_mph"),

    #pickup hr
    pl.col("tpep_pickup_datetime").dt.hour().alias("pickup_hour"),

    #pickup day
    pl.col("tpep_pickup_datetime").dt.weekday().alias("pickup_day_of_week")
])

print(df_feat.select(["trip_duration_minutes", "trip_speed_mph", "pickup_hour", "pickup_day_of_week"]).head(5))


shape: (5, 4)
┌───────────────────────┬────────────────┬─────────────┬────────────────────┐
│ trip_duration_minutes ┆ trip_speed_mph ┆ pickup_hour ┆ pickup_day_of_week │
│ ---                   ┆ ---            ┆ ---         ┆ ---                │
│ f64                   ┆ f64            ┆ i8          ┆ i8                 │
╞═══════════════════════╪════════════════╪═════════════╪════════════════════╡
│ 19.8                  ┆ 5.212121       ┆ 0           ┆ 1                  │
│ 6.6                   ┆ 16.363636      ┆ 0           ┆ 1                  │
│ 17.916667             ┆ 15.739535      ┆ 0           ┆ 1                  │
│ 8.3                   ┆ 10.120482      ┆ 0           ┆ 1                  │
│ 6.1                   ┆ 7.868852       ┆ 0           ┆ 1                  │
└───────────────────────┴────────────────┴─────────────┴────────────────────┘


In [9]:
#save cleaned data

os.makedirs("data/processed", exist_ok=True)

df_feat.write_parquet("data/processed/yellow_tripdata_2024_01_cleaned.parquet")

print("Processed dataset saved.")


Processed dataset saved.


## SQL Analysis Using DuckDB

I used DuckDB to perform SQL-based analysis on the cleaned dataset.  
This allows aggregation, grouping and joining with the zone lookup table.

The following queries explore pickup patterns, fare behavior, payment methods and route popularity.

In [10]:
import duckdb
import polars as pl

#load cleaned data and the zone lookup
df_sql = pl.read_parquet("data/processed/yellow_tripdata_2024_01_cleaned.parquet")
zones = pl.read_csv("data/raw/taxi_zone_lookup.csv")

con = duckdb.connect()

#register both as DuckDB tables
con.register("trips", df_sql.to_pandas())
con.register("zones", zones.to_pandas())

print("DuckDB tables registered: trips, zones")


DuckDB tables registered: trips, zones


### Query 1: Top 10 busiest pickup zones
This query counts how many trips start in each pickup zone and returns the top 10.


In [11]:
q1 = """
SELECT
  z.Zone AS pickup_zone,
  COUNT(*) AS trip_count
FROM trips t
JOIN zones z
  ON t.PULocationID = z.LocationID
GROUP BY z.Zone
ORDER BY trip_count DESC
LIMIT 10;
"""
con.execute(q1).df()


,pickup_zone,trip_count
0,Midtown Center,140161
1,Upper East Side South,140134
2,JFK Airport,138478
3,Upper East Side North,133975
4,Midtown East,104356
5,Times Sq/Theatre District,102972
6,Penn Station/Madison Sq West,102161
7,Lincoln Square East,101800
8,LaGuardia Airport,87715
9,Upper West Side South,86475


### Query 2: Average fare per pickup hour
This query groups trips by pickup hour and calculates the average fare.


In [12]:
q2 = """
SELECT
  pickup_hour,
  AVG(fare_amount) AS avg_fare
FROM trips
GROUP BY pickup_hour
ORDER BY pickup_hour;
"""
con.execute(q2).df()


,pickup_hour,avg_fare
0,0,19.679250
1,1,17.732032
2,2,16.621723
3,3,18.530033
4,4,23.435229
5,5,27.492713
6,6,22.026585
7,7,18.749879
8,8,17.822939
9,9,17.943989


### Query 3: Percentage of trips by payment type
This query calculates how trips are split across payment types as percentages.


In [13]:
q3 = """
SELECT
  payment_type,
  COUNT(*) AS trip_count,
  ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM trips), 2) AS pct_trips
FROM trips
GROUP BY payment_type
ORDER BY trip_count DESC;
"""
con.execute(q3).df()


,payment_type,trip_count,pct_trips
0,1,2298412,80.08
1,2,422921,14.74
2,0,115245,4.02
3,4,22876,0.80
4,3,10648,0.37


### Query 4: Average tip percentage by day of week (credit card only)
This query filters the credit card trips and computes average tip percentage by weekday.


In [14]:
q4 = """
SELECT
  pickup_day_of_week,
  AVG(CASE WHEN fare_amount > 0 THEN (tip_amount / fare_amount) * 100 END) AS avg_tip_pct
FROM trips
WHERE payment_type = 1
GROUP BY pickup_day_of_week
ORDER BY pickup_day_of_week;
"""
con.execute(q4).df()


,pickup_day_of_week,avg_tip_pct
0,1,25.513977
1,2,25.729989
2,3,25.706582
3,4,29.734458
4,5,25.595719
5,6,26.293897
6,7,25.100984


### Query 5: Top 5 pickup-dropoff zone pairs
This query finds the most common route pairs (pickup zone to dropoff zone).


In [15]:
q5 = """
SELECT
  zp.Zone AS pickup_zone,
  zd.Zone AS dropoff_zone,
  COUNT(*) AS trip_count
FROM trips t
JOIN zones zp ON t.PULocationID = zp.LocationID
JOIN zones zd ON t.DOLocationID = zd.LocationID
GROUP BY pickup_zone, dropoff_zone
ORDER BY trip_count DESC
LIMIT 5;
"""
con.execute(q5).df()


,pickup_zone,dropoff_zone,trip_count
0,Upper East Side South,Upper East Side North,21642
1,Upper East Side North,Upper East Side South,19199
2,Upper East Side North,Upper East Side North,15200
3,Upper East Side South,Upper East Side South,14119
4,Midtown Center,Upper East Side South,10139


## Final Insights

From this analysis, several patterns emerge:
Midtown and airport zones dominate total pickup volume.
Average fares increase during peak commuting hours.
Most taxi trips are short-distance rides.
Credit card payments are significantly more common than cash.
Demand varies by day and hour, with weekday peaks in afternoon periods.

Overall, NYC taxi activity is strongly influenced by commuting behavior, commercial hubs, and airport travel.

## AI Tools Used

For this assignment, I used ChatGPT to assist with:
Debugging Python errors
Understanding how to structure the data pipeline
Clarifying SQL syntax
Improving the clarity of written explanations for the markdowns and in streamlit for the diagrams.

The final submission reflects my understanding of the concepts and implementation. 